<a href="https://colab.research.google.com/github/emersonheto/laboratorio_2026/blob/main/Notebook_Treinamento_e_Otimiza%C3%A7%C3%A3o_de_%C3%81rvores_de_Decis%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Programa de Pós-Graduação em Computação - INF/UFRGS**
### Disciplina CMP263 - Aprendizagem de Máquina
#### *Profa. Mariana Recamonde-Mendoza (mrmendoza@inf.ufrgs.br)*
<br>

---
***Observação:*** *Este notebook é disponibilizado aos alunos como complemento às aulas síncronas e aos slides preparados pela professora. Desta forma, os principais conceitos são apresentados no material teórico fornecido. *

---

# Otimização de Hiperparâmetros e Seleção de Modelos com Árvores de Decisão

Neste notebook, exploraremos o impacto da escolha de hiperparâmetros no desempenho de modelos de árvores de decisão.

Focaremos em:

- Pré-poda (`max_depth`)
- Pós-poda (`ccp_alpha`)
- Avaliação com holdout (70/15/15)

Também discutiremos:
- Bias vs Variance
- Underfitting
- Overfitting


### Viés, Variância e Complexidade em Árvores de Decisão

Modelos de aprendizado de máquina estão sujeitos ao chamado tradeoff entre viés e variância. Árvores de decisão muito simples (rasas) tendem a apresentar alto viés, não conseguindo capturar padrões relevantes dos dados (underfitting). Por outro lado, árvores muito profundas tendem a apresentar alta variância, ajustando-se excessivamente aos dados de treinamento e capturando ruído (overfitting).

Neste notebook, exploramos como hiperparâmetros como `max_depth` (pré-poda) e `ccp_alpha` (pós-poda) permitem controlar a complexidade do modelo. O objetivo é encontrar um equilíbrio entre viés e variância que maximize a capacidade de generalização do modelo para dados não vistos.

## Importação de bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


## Carregamento dos dados

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names
class_names = data.target_names

n_malign = np.sum(y == 0) # Atenção: no dataset do scikit-learn, o maligno está mapeado como 0
n_benign = np.sum(y == 1)


print("Número de exemplos malignos: %d" % n_malign)
print("Número de exemplos benignos: %d" % n_benign)

In [ ]:
print(feature_names)

In [ ]:
len(feature_names) ##número de atributos no dataset

## Função auxiliar para avaliar o modelo (opcional)

Essa avaliação poderia ser feita sem a função, mas encapsular as operações na função simplifica sua chamada.

In [ ]:
def avaliar_modelo(modelo, X_train, y_train, X_val, y_val, X_test, y_test):
    modelo.fit(X_train, y_train)

    return (
        accuracy_score(y_train, modelo.predict(X_train)),
        accuracy_score(y_val, modelo.predict(X_val)),
        accuracy_score(y_test, modelo.predict(X_test))
    )

# Parte 1: Analisando a estrutura e características das árvores de decisão

Como estudado em aula, a árvore de decisão é conhecida por ser um classificador com alta variância. Isso possui consequências na estrutura das árvores treinadas.

O código abaixo treina várias árvores de decisão com diferentes conjuntos de treino obtidos através do método holdout.
**Use-o para responder à Questão 1 do questionário.**


In [ ]:
def get_root_node(dt, feature_names):
    feature_idx = dt.tree_.feature[0]
    return feature_names[feature_idx]


n_repeats = 20
root_nodes = []

# variando o seed do holdout, geramos conjuntos de treino e teste um pouco diferentes a cada iteração
for split_random_state in range(0, n_repeats):
  # Holdout com 20% de dados de teste
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=split_random_state)

  # Treinamento da árvore usando os dados de treino
  dt = DecisionTreeClassifier(random_state=0)
  dt.fit(X_train, y_train)

  # Obtemos o atributo usado na raiz e o salvamos na lista
  root_node = get_root_node(dt, feature_names)
  root_nodes.append(root_node)

root_nodes

Podemos observar a estrutura da árvore de decisão treinada (neste caso, sem nenhuma poda). Por exemplo, aqui temos a última árvore treinada no processo iterativo da célula anterior.

Quantos atributos são efetivamente utilizados na árvore treinada? **Observe esta característica e responda à Questão 2 do questionário**

In [ ]:
#Paea visualizar a estrutura da árvore
import graphviz
from sklearn import tree

dot_data = tree.export_graphviz(dt,
                                out_file=None,
                                feature_names = feature_names,
                                class_names= class_names,
                                filled=True)

## Plotar a árvore de decisão no notebook
graph = graphviz.Source(dot_data)
graph

## Para salvar como png, descomente as linhas abaixo
#graph.format = 'png'
#graph.render('DecisionTree1',view = True)

In [ ]:
import pandas as pd

features_used = dt.tree_.feature
features_used = features_used[features_used >= 0]

print(feature_names[features_used])

feature_counts = pd.Series(features_used).value_counts()

feature_usage = pd.DataFrame({
    'Feature': [feature_names[i] for i in feature_counts.index],
    'Nº de divisões': feature_counts.values
})

feature_usage

# Parte 2: Otimização com um Holdout único

In [ ]:
# Proporções desejadas
test_size = 0.15
val_size_total = 0.15  # em relação ao total

# Primeiro: separa teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

# Ajuste da validação em relação ao conjunto restante
val_size_adjusted = val_size_total / (1 - test_size)

# Segundo: separa treino e validação
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size_adjusted, random_state=42, stratify=y_temp
)

# --- Análise das proporções ---
n_total = len(X)
n_train = len(X_train)
n_val = len(X_val)
n_test = len(X_test)

print("Tamanho das partições:")
print(f"Treino: {n_train} ({n_train/n_total:.2%})")
print(f"Validação: {n_val} ({n_val/n_total:.2%})")
print(f"Teste: {n_test} ({n_test/n_total:.2%})")

Observe que a proporção de validação precisa ser ajustada após a separação do conjunto de teste.

Como primeiro removemos 15% dos dados para teste, os 15% de validação devem ser calculados em relação aos 85% restantes. Isso garante que, ao final, as proporções sejam aproximadamente 70% treino, 15% validação e 15% teste.

## Pré-poda: max_depth

Neste trecho, investigamos o impacto do hiperparâmetro `max_depth` — responsável por controlar a profundidade máxima da árvore de decisão — sobre o desempenho do modelo.

Para isso, treinamos múltiplos modelos variando `max_depth` de 1 a 20. Para cada valor, o modelo é ajustado com os dados de treino e avaliado tanto no conjunto de treino quanto no conjunto de validação. As acurácias obtidas são armazenadas separadamente, permitindo analisar como o desempenho evolui à medida que aumentamos a complexidade do modelo.

Ao final, construímos um gráfico comparando as curvas de desempenho em treino e validação. Em geral, espera-se que a acurácia em treino aumente monotonamente com a profundidade, enquanto a acurácia em validação tende a atingir um máximo e depois estagnar ou diminuir, evidenciando o fenômeno de overfitting. Esse tipo de análise é fundamental para escolher um valor de `max_depth` que proporcione boa capacidade de generalização.

**Observe o gráfico gerado abaixo, e responda às Questões 3 e 4 do questionário.**

In [ ]:
depths = range(1, 21)
train_scores = []
val_scores = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    train_acc, val_acc, _ = avaliar_modelo(model, X_train, y_train, X_val, y_val, X_test, y_test)

    train_scores.append(train_acc)
    val_scores.append(val_acc)

plt.plot(depths, train_scores, label='Treino')
plt.plot(depths, val_scores, label='Validação')
plt.xlabel('max_depth')
plt.ylabel('Acurácia')
plt.legend()
plt.show()


**Discussão:**

- Baixa profundidade → underfitting (alto viés)
- Alta profundidade → overfitting (alta variância)


## Pós-poda: cost-complexity pruning

In [ ]:
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas
print(alphas)

train_scores = []
val_scores = []

for alpha in alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    train_acc, val_acc, _ = avaliar_modelo(model, X_train, y_train, X_val, y_val, X_test, y_test)

    train_scores.append(train_acc)
    val_scores.append(val_acc)

plt.plot(alphas, train_scores, label='Treino')
plt.plot(alphas, val_scores, label='Validação')
plt.xlabel('ccp_alpha')
plt.ylabel('Acurácia')
plt.legend()
plt.show()


Neste trecho, exploramos a **pós-poda** em árvores de decisão por meio do parâmetro `ccp_alpha`. Inicialmente, utilizamos o método `cost_complexity_pruning_path` para obter uma sequência de valores candidatos de `ccp_alpha`, que representam diferentes níveis de simplificação da árvore.

Para cada valor de `ccp_alpha`, treinamos um modelo e avaliamos sua acurácia nos conjuntos de treino e validação. À medida que `ccp_alpha` aumenta, a árvore se torna mais simples (mais podada). O gráfico resultante permite visualizar como o desempenho varia com a complexidade do modelo, ajudando a identificar um valor de `ccp_alpha` que equilibre complexidade e capacidade de generalização.

**Observe o gráfico gerado abaixo e responda à Questão 5 do questionário.**

Para mais informações:

https://scikit-learn.org/stable/modules/tree.html#minimal-cost-complexity-pruning
https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html

In [ ]:
from sklearn.tree import plot_tree

# Defina um valor de ccp_alpha (por exemplo, o melhor encontrado)
alpha_escolhido = 0.00447803 # ajuste conforme necessário! Este é apenas um exemplo

# Treinar modelo com esse alpha
model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha_escolhido)
model.fit(X_train, y_train)

# Plotar a árvore
plt.figure(figsize=(12, 6))
plot_tree(
    model,
    filled=True,
    feature_names=feature_names,
    class_names=class_names,
    rounded=True
)
plt.title(f"Árvore de decisão (ccp_alpha = {alpha_escolhido})")
plt.show()

# Parte 3: Repetição do holdout (20 vezes)

Nesta parte, vamos observar como o desempenho do modelo treinado varia para diferentes partições de treino e validação, independente do mecanismo de poda.

**Utilize os resultados gerados nesta seção para responder à Questão 6 do questionário.**

In [ ]:
n_reps = 20
depths = range(1, 21)

# d -> lista de resultados
results = {d: [] for d in depths}

for i in range(n_reps):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp)

    for d in depths:
        model = DecisionTreeClassifier(max_depth=d)
        _, val_acc, _ = avaliar_modelo(
            model, X_train, y_train, X_val, y_val, X_test, y_test
        )
        results[d].append(val_acc)

# Preparar dados para o boxplot
data = [results[d] for d in depths]

plt.figure(figsize=(10,5))
plt.boxplot(data)
plt.xticks(range(1, len(depths)+1), depths)
plt.xlabel('max_depth')
plt.ylabel('Acurácia de validação')
plt.title('Variação de desempenho para diferentes valores de max_depth')
plt.show()

Observe que agora não analisamos apenas o melhor modelo, mas a distribuição de desempenho para cada valor de `max_depth`.

Isso permite visualizar a variância associada a cada nível de complexidade do modelo. Em geral, modelos mais complexos (maior profundidade) tendem a apresentar maior variabilidade no desempenho, evidenciando maior sensibilidade à divisão dos dados. Entretanto, árvorees de decisão são modelos muito sensíveis à varuiação nos dados, apresentando dispersão mesmo em modelos mais simples.

In [ ]:
n_reps = 20
depths = range(1, 21)

train_results = {d: [] for d in depths}
val_results = {d: [] for d in depths}

for i in range(n_reps):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp)

    for d in depths:
        model = DecisionTreeClassifier(max_depth=d)
        train_acc, val_acc, _ = avaliar_modelo(
            model, X_train, y_train, X_val, y_val, X_test, y_test
        )

        train_results[d].append(train_acc)
        val_results[d].append(val_acc)

# calcular médias e desvios
train_means = [np.mean(train_results[d]) for d in depths]
train_stds  = [np.std(train_results[d]) for d in depths]

val_means = [np.mean(val_results[d]) for d in depths]
val_stds  = [np.std(val_results[d]) for d in depths]

# plot
plt.figure(figsize=(10,5))

plt.plot(depths, train_means, label='Treino')
plt.fill_between(depths,
                 np.array(train_means) - np.array(train_stds),
                 np.array(train_means) + np.array(train_stds),
                 alpha=0.2)

plt.plot(depths, val_means, label='Validação')
plt.fill_between(depths,
                 np.array(val_means) - np.array(val_stds),
                 np.array(val_means) + np.array(val_stds),
                 alpha=0.2)

plt.xlabel('max_depth')
plt.ylabel('Acurácia')
plt.title('Média e variabilidade: Treino vs Validação')
plt.legend()
plt.show()

Este gráfico mostra a média de desempenho ao longo de múltiplas execuções, juntamente com a variabilidade (desvio padrão).

Observe que:

- Para valores baixos de `max_depth`, treino e validação têm desempenhos próximos → modelo simples (alto viés).
- À medida que a profundidade aumenta, o desempenho em treino cresce continuamente.
- Já a validação tende a estabilizar ou cair → indicando overfitting.
- A região ideal está próxima ao ponto onde a curva de validação atinge seu máximo e/ou não se distancia muito do desempenho em treinamento

As faixas sombreadas indicam a variabilidade do modelo: modelos mais complexos tendem a apresentar maior instabilidade.

# Discussão Final

Ao longo deste notebook, exploramos como a escolha de hiperparâmetros influencia diretamente a complexidade e o desempenho de árvores de decisão. Observamos que tanto a pré-poda (`max_depth`) quanto a pós-poda (`ccp_alpha`) atuam como mecanismos de controle da complexidade do modelo, permitindo mitigar problemas de underfitting e overfitting.

A análise com validação mostrou que existe um ponto intermediário de complexidade que maximiza a capacidade de generalização. Além disso, ao repetir o processo de holdout, evidenciamos a variabilidade do desempenho, reforçando a importância de avaliar modelos de forma mais robusta.

Até este ponto, utilizamos o conjunto de validação para selecionar o melhor hiperparâmetro. No entanto, o conjunto de teste ainda não foi utilizado e deve permanecer “intocado” até a etapa final.

O próximo passo consiste em **treinar um novo modelo utilizando todos os dados disponíveis para treinamento (treino + validação)**, agora fixando o melhor valor de `max_depth`, e então avaliar seu desempenho no conjunto de teste. Essa etapa fornece uma estimativa mais fiel da performance do modelo em dados não vistos.

# Treinando e avaliando o modelo final (sua vez!)

Reutilize as definições de variáveis do notebook e o melhor valor de max_depth (ou ccp_alpha)
encontrado. Treine o modelo utilizando todos os dados de treinamento e validação, e avalie o resultado no conjunto de teste.

In [ ]:
# ==========================================
# Etapa final: treinamento e avaliação no teste
# ==========================================

# 1. Identificar o melhor valor de max_depth (ou ccp_alpha)
# (assumindo que val_means foi calculado anteriormente)

# 2. Reunir treino + validação


# 3. Treinar modelo final com o melhor hiperparâmetro


# 4. Avaliar no conjunto de teste (nunca usado antes!)